# 02 — Build SILVER & GOLD Layers

This notebook creates the analytics pipeline:
1. **SILVER Dynamic Tables** — auto-refreshing cleaned data (5-min lag)
2. **GOLD Views** — star schema for analytics


In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE WAREHOUSE HYDRAB_HOL_WH;

## SILVER: Dynamic Tables

Dynamic Tables auto-refresh from BRONZE. We set a 5-minute target lag.
This works because BRONZE contains native tables (not shared views).


In [ ]:
-- Vehicles Silver: clean vehicle master
CREATE OR REPLACE DYNAMIC TABLE SILVER.VEHICLES_SILVER
  TARGET_LAG = '5 minutes'
  WAREHOUSE = HYDRAB_HOL_WH
AS
SELECT
  Chassis_Number__c AS vin,
  Name AS model,
  AccountId AS customer_id,
  Bus_Depot__c AS depot_id,
  Status AS status,
  Production_Status__c AS production_status,
  Delivery_Date__c AS delivery_date,
  Registration__c AS registration
FROM BRONZE.ASSET
WHERE Chassis_Number__c IS NOT NULL;

In [ ]:
-- Telemetry Silver: parsed Odos events
CREATE OR REPLACE DYNAMIC TABLE SILVER.TELEMETRY_SILVER
  TARGET_LAG = '5 minutes'
  WAREHOUSE = HYDRAB_HOL_WH
AS
SELECT
  RAW:vin::STRING AS vin,
  RAW:timestamp::TIMESTAMP AS event_time,
  RAW:signals:battery_soc::FLOAT AS battery_soc,
  RAW:signals:speed::FLOAT AS speed_kmh,
  RAW:signals:battery_temperature::FLOAT AS cell_temp,
  RAW:signals:latitude::FLOAT AS latitude,
  RAW:signals:longitude::FLOAT AS longitude,
  RAW:signals:odometer::FLOAT AS odometer_km,
  RAW:signals:energy_consumed::FLOAT AS energy_kwh
FROM BRONZE.ODOS_EVENTS;

In [ ]:
-- Defects Silver
CREATE OR REPLACE DYNAMIC TABLE SILVER.DEFECTS_SILVER
  TARGET_LAG = '5 minutes'
  WAREHOUSE = HYDRAB_HOL_WH
AS
SELECT
  "Asset__c" AS asset_id,
  "Defect_Type__c" AS defect_type,
  "Severity__c" AS severity,
  "Root_Cause__c" AS root_cause,
  "Repair_Cost__c" AS repair_cost,
  "CreatedDate" AS created_date
FROM BRONZE.DEFECT_EVENT;

## GOLD: Star Schema Views

In [ ]:
USE SCHEMA GOLD;

-- Dimension: Vehicle
CREATE OR REPLACE VIEW DIM_VEHICLE AS
SELECT * FROM SILVER.VEHICLES_SILVER;

-- Fact: Latest Telemetry per Vehicle
CREATE OR REPLACE VIEW FCT_LATEST_TELEMETRY AS
SELECT *
FROM SILVER.TELEMETRY_SILVER
QUALIFY ROW_NUMBER() OVER (PARTITION BY vin ORDER BY event_time DESC) = 1;

-- Fact: Defects
CREATE OR REPLACE VIEW FCT_DEFECT AS
SELECT * FROM SILVER.DEFECTS_SILVER;

## SILVER & GOLD layers complete!

- 3 Dynamic Tables in SILVER (auto-refresh every 5 minutes)
- 3 GOLD views (DIM_VEHICLE, FCT_LATEST_TELEMETRY, FCT_DEFECT)

Next: notebook 03 creates the Semantic View and Cortex Agent.